# IPSAE Interface Scoring

![IPSAE Interface Scoring](https://proto-bio.github.io/proto-assets/images/tool/ipsae/hero.png)

This notebook demonstrates `run_ipsae_scoring`, which scores a cofolded protein complex using the IPSAE method (Dunbrack 2025). It computes ipSAE, pDockQ2, LIS, pDockQ, and ipTM metrics from per-residue pLDDT and the PAE matrix.

The tool requires a `Structure` with pLDDT in the B-factor column and a PAE matrix at `structure.metrics['pae']`.

In [1]:
from pathlib import Path

from proto_tools.tools.structure_scoring.ipsae import (
    IPSAEScoringConfig,
    IPSAEScoringInput,
    run_ipsae_scoring,
)
from proto_tools.entities.structures import Structure
from proto_tools.entities.structures.structure import BFactorType

In [2]:
import json

# Bundled barnase-barstar complex (chains A/B) cofolded by Boltz-2, with its real PAE matrix.
pdb_path = Path("..") / "example_input_fixture.pdb"
pae = json.loads((Path("..") / "example_input_fixture_pae.json").read_text())

structure = Structure.from_file(
    pdb_path, b_factor_type=BFactorType.PLDDT, metrics={"pae": pae}
)

inputs = IPSAEScoringInput(
    structure=structure, binder_chain="A", target_chains=["B"]
)
print(f"Chains: {structure.get_chain_ids()}")
print(f"Binder: A | Target: B")

Chains: ['A', 'B']
Binder: A | Target: B


In [3]:
# Run IPSAE scoring with default config
config = IPSAEScoringConfig()
result = run_ipsae_scoring(inputs, config)

print(f"ipsae:      {result.metrics.ipsae:.4f}")
print(f"pdockq2:    {result.metrics.pdockq2:.4f}")
print(f"lis:        {result.metrics.lis:.4f}")
print(f"pdockq:     {result.metrics.pdockq:.4f}")
print(f"iptm_d0chn: {result.metrics.iptm_d0chn:.4f}")

Running run_ipsae_scoring [00:00]

ipsae:      0.9323
pdockq2:    0.9381
lis:        0.7737
pdockq:     0.4845
iptm_d0chn: 0.9617


In [4]:
# Inspect per-chain-pair breakdown
for row in result.metrics.chain_pair_results:
    print(
        f"{row.chain1}-{row.chain2} ({row.pair_type:4s}) | "
        f"ipSAE={row.ipsae:.4f}  pDockQ2={row.pdockq2:.4f}  "
        f"LIS={row.lis:.4f}  pDockQ={row.pdockq:.4f}  ipTM={row.iptm_d0chn:.4f}"
    )

A-B (asym) | ipSAE=0.8794  pDockQ2=0.9343  LIS=0.7870  pDockQ=0.4845  ipTM=0.9443
B-A (asym) | ipSAE=0.9323  pDockQ2=0.9381  LIS=0.7604  pDockQ=0.4845  ipTM=0.9617
A-B (max ) | ipSAE=0.9323  pDockQ2=0.9381  LIS=0.7737  pDockQ=0.4845  ipTM=0.9617
